In [1]:
import warnings, time
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

# LogisticRegression
from sklearn.linear_model import LogisticRegression

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity


# add path to project root
import sys
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))
if PROJECT_ROOT.name == "notebooks":
    sys.path.append(str(PROJECT_ROOT.parent))

from social_media_jigsaw.models import preprocessing

pd.set_option('display.max_columns', None)
plt.rcParams.update({'figure.dpi': 100})

In [2]:
# config
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'processed_data').exists() and PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label',
    'cv_folds': 3,
    'scoring': 'f1_macro',
    'tfidf': dict(
        ngram_range=(1, 2),
        min_df=1,
        max_df=1.0,
        sublinear_tf=True,
        norm='l2',
    ),
    'class_map': {
        0: 'Non-Toxic',
        1: 'Insults/Flaming',
        2: 'Other Offensive',
        3: 'Hate/Harassment',
        4: 'Threats',
        5: 'Extremism',
    },
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

print('CONFIG loaded. Text column:', tc)


CONFIG loaded. Text column: message


In [3]:

# load WOT train data
d = CONFIG['data_dir']

wot_train_x = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train_y = wot_train_x[lc]
wot_train_y_binary = (wot_train_y != 0).astype(int)

wot_text = preprocessing.preprocess_df(wot_train_x[tc])

print('WOT train shape:', wot_train_x.shape)
print('WOT train class distribution:\n', wot_train_y.value_counts())
print('WOT train binary distribution:\n', wot_train_y_binary.value_counts())


# load the validation sets for WOT and DOTA
wot_val_x = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val_y = wot_val_x[lc]
wot_val_y_binary = (wot_val_y != 0).astype(int)

wot_val_text = preprocessing.preprocess_df(wot_val_x[tc])

WOT train shape: (31401, 2)
WOT train class distribution:
 label
0.0    24942
1.0     4841
2.0     1328
3.0      212
4.0       54
5.0       24
Name: count, dtype: int64
WOT train binary distribution:
 label
0    24942
1     6459
Name: count, dtype: int64


In [4]:

# load DOTA training data
dota_train_x = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train_y = dota_train_x[lc]
dota_train_y_binary = (dota_train_y != 0).astype(int)

dota_text = preprocessing.preprocess_df(dota_train_x[tc])

print('DOTA training shape:', dota_train_x.shape)
print('DOTA training class distribution:\n', dota_train_y.value_counts())
print('DOTA training binary distribution:\n', dota_train_y_binary.value_counts())

# load DOTA validation data
dota_val_x = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val_y = dota_val_x[lc]
dota_val_y_binary = (dota_val_y != 0).astype(int)

dota_val_text = preprocessing.preprocess_df(dota_val_x[tc])

DOTA training shape: (22306, 2)
DOTA training class distribution:
 label
0    15987
1     3518
2     1478
3     1323
Name: count, dtype: int64
DOTA training binary distribution:
 label
0    15987
1     6319
Name: count, dtype: int64


In [5]:

# load social media models 
import joblib

social_media_models = {}
for model_file in (CONFIG['model_dir'] / 'binary' ).glob('social_media_*.joblib'):
    print(f'Loading model from {model_file}...')
    model_name = model_file.stem
    social_media_models[model_name] = joblib.load(model_file)
    print(f'Loaded model: {model_name} from {model_file}')
    

# load the fitted scaler and nb_tfidf 
scaler = joblib.load(CONFIG['model_dir'] / 'helper' / 'social_media_max_abs_scaler.joblib')
nb_tfidf = joblib.load(CONFIG['model_dir'] / 'helper' / 'social_media_nb_tfidf.joblib')

Loading model from /Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_ComplementNB_model.joblib...
Loaded model: social_media_ComplementNB_model from /Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_ComplementNB_model.joblib
Loading model from /Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_LinearSVC_(Optuna)_model.joblib...
Loaded model: social_media_LinearSVC_(Optuna)_model from /Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_LinearSVC_(Optuna)_model.joblib
Loading model from /Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_LogisticRegressionCV_model.joblib...


Loaded model: social_media_LogisticRegressionCV_model from /Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_LogisticRegressionCV_model.joblib


In [6]:
# convert text to features using the fitted scaler and nb_tfidf
wot_train_nb = nb_tfidf.transform(wot_text)
wot_train_nb_scaled = scaler.transform(wot_train_nb)

dota_train_nb = nb_tfidf.transform(dota_text)
dota_train_nb_scaled = scaler.transform(dota_train_nb)

# convert the validation sets to features as well
wot_val_nb = nb_tfidf.transform(wot_val_text)
wot_val_nb_scaled = scaler.transform(wot_val_nb)

dota_val_nb = nb_tfidf.transform(dota_val_text)
dota_val_nb_scaled = scaler.transform(dota_val_nb)

In [7]:
def _safe_confusion_rates(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'FPR': fp / (fp + tn) if (fp + tn) > 0 else 0,
        'FNR': fn / (fn + tp) if (fn + tp) > 0 else 0,
        'TPR': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'TNR': tn / (tn + fp) if (tn + fp) > 0 else 0,
    }
    
    
def precision(y_true, y_pred):
    rates = _safe_confusion_rates(y_true, y_pred)
    return rates['TPR'] / (rates['TPR'] + rates['FPR']) if (rates['TPR'] + rates['FPR']) > 0 else 0

def _prediction_metrics(y_true, y_pred):
    return {
        'CV Macro F1': f1_score(y_true, y_pred, average='macro'),
        'CV Weighted F1': f1_score(y_true, y_pred, average='weighted'),
        'Accuracy': accuracy_score(y_true, y_pred),
        "Precision": precision(y_true, y_pred),
        **_safe_confusion_rates(y_true, y_pred),
    }

In [8]:
from scipy.sparse import vstack

# evaluate social media models on WOT, DOTA, and combined validation sets

res = {
    "model_name": [],
    "validation_set_name": [],
    "CV Macro F1": [],
    "CV Weighted F1": [],
    "Accuracy": [],
    "Precision": [],
    "FPR": [],
    "FNR": [],
    "TPR": [],
    "TNR": [],
}

validation_sets_scaled = {
    "WOT": (
        vstack([wot_train_nb_scaled]),
        wot_train_y_binary
    ),
    "DOTA": (
        vstack([dota_train_nb_scaled]),
        dota_train_y_binary
    ),
    "COMBINED": (
        vstack([wot_train_nb_scaled, dota_train_nb_scaled]),
        np.hstack([wot_train_y_binary, dota_train_y_binary])
    ),
}

validation_sets_nb = {
    "WOT-TF-IDF-ONLY": (
        vstack([wot_train_nb]),
        wot_train_y_binary
    ),
    "DOTA-TF-IDF-ONLY": (
        vstack([dota_train_nb]),
        dota_train_y_binary
    ),
    "COMBINED-TF-IDF-ONLY": (
        vstack([wot_train_nb, dota_train_nb]),
        np.hstack([wot_train_y_binary, dota_train_y_binary])
    ),
}

for model_name, model in social_media_models.items():

    if "ComplementNB" in model_name:
        validation_sets = validation_sets_nb
    else:
        validation_sets = validation_sets_scaled

    for validation_set_name, (X_val, y_val) in validation_sets.items():
        print(f"Evaluating model {model_name} on validation set {validation_set_name}...")

        y_pred = model.predict(X_val)
        metrics = _prediction_metrics(y_val, y_pred)

        res["model_name"].append(model_name)
        res["validation_set_name"].append(validation_set_name)

        for metric_name, metric_value in metrics.items():
            res[metric_name].append(metric_value)

results_df = pd.DataFrame(res)

Evaluating model social_media_ComplementNB_model on validation set WOT-TF-IDF-ONLY...
Evaluating model social_media_ComplementNB_model on validation set DOTA-TF-IDF-ONLY...
Evaluating model social_media_ComplementNB_model on validation set COMBINED-TF-IDF-ONLY...
Evaluating model social_media_LinearSVC_(Optuna)_model on validation set WOT...
Evaluating model social_media_LinearSVC_(Optuna)_model on validation set DOTA...
Evaluating model social_media_LinearSVC_(Optuna)_model on validation set COMBINED...
Evaluating model social_media_LogisticRegressionCV_model on validation set WOT...
Evaluating model social_media_LogisticRegressionCV_model on validation set DOTA...
Evaluating model social_media_LogisticRegressionCV_model on validation set COMBINED...


In [9]:
results_df.sort_values(by=['CV Macro F1', 'CV Weighted F1', 'Accuracy'], ascending=False, inplace=True)
results_df

,model_name,validation_set_name,CV Macro F1,CV Weighted F1,Accuracy,Precision,FPR,FNR,TPR,TNR
3,social_media_LinearSVC_(Optuna)_model,WOT,0.758345,0.843587,0.845132,0.868140,0.090771,0.402384,0.597616,0.909229
6,social_media_LogisticRegressionCV_model,WOT,0.750263,0.837312,0.837840,0.856766,0.099832,0.402849,0.597151,0.900168
5,social_media_LinearSVC_(Optuna)_model,COMBINED,0.732013,0.808306,0.811067,0.834699,0.111510,0.436923,0.563077,0.888490
8,social_media_LogisticRegressionCV_model,COMBINED,0.721943,0.800600,0.802931,0.822841,0.119109,0.446784,0.553216,0.880891
4,social_media_LinearSVC_(Optuna)_model,DOTA,0.698087,0.758817,0.763113,0.785798,0.143867,0.472227,0.527773,0.856133
7,social_media_LogisticRegressionCV_model,DOTA,0.685568,0.749048,0.753788,0.773102,0.149184,0.491692,0.508308,0.850816
0,social_media_ComplementNB_model,WOT-TF-IDF-ONLY,0.669180,0.757852,0.737779,0.733877,0.248938,0.313516,0.686484,0.751062
2,social_media_ComplementNB_model,COMBINED-TF-IDF-ONLY,0.649935,0.718770,0.699201,0.697664,0.295121,0.318986,0.681014,0.704879
1,social_media_ComplementNB_model,DOTA-TF-IDF-ONLY,0.618675,0.662013,0.644894,0.647828,0.367173,0.324577,0.675423,0.632827


In [10]:
import numpy as np
import pandas as pd

def evaluate_social_media_majority_ensemble(
    social_media_models,
    validation_sets_scaled,
    validation_sets_nb,
):
    res = {
        "ensemble_name": [],
        "validation_set_name": [],
        "CV Macro F1": [],
        "CV Weighted F1": [],
        "Accuracy": [],
        "Precision": [],
        "FPR": [],
        "FNR": [],
        "TPR": [],
        "TNR": [],
    }

    for validation_set_name in ["WOT", "DOTA", "COMBINED"]:

        X_scaled, y_val = validation_sets_scaled[validation_set_name]
        X_nb, _ = validation_sets_nb[f"{validation_set_name}-TF-IDF-ONLY"]

        preds = []

        for model_name, model in social_media_models.items():

            if "ComplementNB" in model_name:
                y_pred = model.predict(X_nb)
            else:
                y_pred = model.predict(X_scaled)

            preds.append(y_pred)

        preds = np.vstack(preds)

        # majority vote: if at least 2 of 3 models predict toxic, predict toxic
        ensemble_pred = (preds.sum(axis=0) >= 2).astype(int)

        metrics = _prediction_metrics(y_val, ensemble_pred)

        res["ensemble_name"].append("Simple Majority Vote")
        res["validation_set_name"].append(validation_set_name)

        for metric_name, metric_value in metrics.items():
            res[metric_name].append(metric_value)

    return pd.DataFrame(res)

In [11]:
ensemble_results_df = evaluate_social_media_majority_ensemble(
    social_media_models=social_media_models,
    validation_sets_scaled=validation_sets_scaled,
    validation_sets_nb=validation_sets_nb,
)

ensemble_results_df.sort_values(by=['CV Macro F1', 'CV Weighted F1', 'Accuracy'], ascending=False, inplace=True)
ensemble_results_df

,ensemble_name,validation_set_name,CV Macro F1,CV Weighted F1,Accuracy,Precision,FPR,FNR,TPR,TNR
0,Simple Majority Vote,WOT,0.754655,0.840054,0.840451,0.859832,0.098709,0.394488,0.605512,0.901291
2,Simple Majority Vote,COMBINED,0.728073,0.804348,0.805947,0.825728,0.120110,0.430897,0.569103,0.879890
1,Simple Majority Vote,DOTA,0.693672,0.754218,0.757375,0.776040,0.153500,0.468112,0.531888,0.846500


In [12]:
import numpy as np
import pandas as pd
from itertools import product

def get_model_predictions_for_validation_set(
    social_media_models,
    validation_set_name,
    validation_sets_scaled,
    validation_sets_nb,
):
    X_scaled, y_val = validation_sets_scaled[validation_set_name]
    X_nb, _ = validation_sets_nb[f"{validation_set_name}-TF-IDF-ONLY"]

    model_names = []
    preds = []

    for model_name, model in social_media_models.items():
        if "ComplementNB" in model_name:
            X_use = X_nb
        else:
            X_use = X_scaled

        y_pred = model.predict(X_use)

        model_names.append(model_name)
        preds.append(y_pred)

    preds = np.vstack(preds).T  # shape: n_samples x n_models

    return model_names, preds, y_val

In [13]:
def tune_weighted_ensemble(
    social_media_models,
    validation_sets_scaled,
    validation_sets_nb,
    validation_set_name="COMBINED",
    weight_step=0.05,
    thresholds=None,
    sort_metric="CV Macro F1",
):
    if thresholds is None:
        thresholds = np.arange(0.10, 0.91, 0.01)

    model_names, preds, y_val = get_model_predictions_for_validation_set(
        social_media_models=social_media_models,
        validation_set_name=validation_set_name,
        validation_sets_scaled=validation_sets_scaled,
        validation_sets_nb=validation_sets_nb,
    )

    n_models = len(model_names)

    if n_models != 3:
        raise ValueError("This tuning function currently assumes exactly 3 models.")

    weight_values = np.arange(0, 1 + weight_step, weight_step)

    results = []

    for w1 in weight_values:
        for w2 in weight_values:
            w3 = 1 - w1 - w2

            if w3 < 0:
                continue

            weights = np.array([w1, w2, w3])

            # avoid floating-point weirdness
            if not np.isclose(weights.sum(), 1.0):
                continue

            scores = preds @ weights

            for threshold in thresholds:
                y_pred = (scores >= threshold).astype(int)

                metrics = _prediction_metrics(y_val, y_pred)

                row = {
                    "validation_set_name": validation_set_name,
                    "threshold": round(float(threshold), 3),
                }

                for model_name, weight in zip(model_names, weights):
                    row[f"weight__{model_name}"] = round(float(weight), 3)

                row.update(metrics)
                results.append(row)

    results_df = pd.DataFrame(results)

    return (
        results_df
        .sort_values(sort_metric, ascending=False)
        .reset_index(drop=True)
    )

In [14]:
tuned_combined_df = tune_weighted_ensemble(
    social_media_models=social_media_models,
    validation_sets_scaled=validation_sets_scaled,
    validation_sets_nb=validation_sets_nb,
    validation_set_name="COMBINED",
    weight_step=0.05,
    thresholds=np.arange(0.10, 0.91, 0.01),
    sort_metric="CV Macro F1",
)

tuned_combined_df.head(10)

KeyboardInterrupt: 

In [ ]:
best_row = tuned_combined_df.iloc[0]

best_threshold = best_row["threshold"]

best_weights = {}

for col in tuned_combined_df.columns:
    if col.startswith("weight__"):
        model_name = col.replace("weight__", "")
        best_weights[model_name] = best_row[col]

best_weights, best_threshold

({'social_media_ComplementNB_model': np.float64(0.35),
  'social_media_LinearSVC_(Optuna)_model': np.float64(0.5),
  'social_media_LogisticRegressionCV_model': np.float64(0.15)},
 np.float64(0.7))

In [ ]:
def evaluate_fixed_weighted_ensemble(
    social_media_models,
    validation_sets_scaled,
    validation_sets_nb,
    weights,
    threshold,
    validation_set_names=("WOT", "DOTA", "COMBINED"),
):
    res = {
        "model_name": [],
        "validation_set_name": [],
        "threshold": [],
        "CV Macro F1": [],
        "CV Weighted F1": [],
        "Accuracy": [],
        "Precision": [],
        "FPR": [],
        "FNR": [],
        "TPR": [],
        "TNR": [],
    }

    for validation_set_name in validation_set_names:
        model_names, preds, y_val = get_model_predictions_for_validation_set(
            social_media_models=social_media_models,
            validation_set_name=validation_set_name,
            validation_sets_scaled=validation_sets_scaled,
            validation_sets_nb=validation_sets_nb,
        )

        weight_vector = np.array([weights[name] for name in model_names])

        scores = preds @ weight_vector
        y_pred = (scores >= threshold).astype(int)

        metrics = _prediction_metrics(y_val, y_pred)

        res["model_name"].append("Tuned Weighted Ensemble")
        res["validation_set_name"].append(validation_set_name)
        res["threshold"].append(threshold)

        for metric_name, metric_value in metrics.items():
            res[metric_name].append(metric_value)

    return pd.DataFrame(res)

In [ ]:
tuned_ensemble_eval_df = evaluate_fixed_weighted_ensemble(
    social_media_models=social_media_models,
    validation_sets_scaled=validation_sets_scaled,
    validation_sets_nb=validation_sets_nb,
    weights=best_weights,
    threshold=best_threshold,
)

tuned_ensemble_eval_df.sort_values(by=['CV Macro F1', 'CV Weighted F1', 'Accuracy'], ascending=False, inplace=True)
tuned_ensemble_eval_df

,model_name,validation_set_name,threshold,CV Macro F1,CV Weighted F1,Accuracy,Precision,FPR,FNR,TPR,TNR
0,Tuned Weighted Ensemble,WOT,0.7,0.765637,0.850411,0.854145,0.885255,0.075495,0.417557,0.582443,0.924505
2,Tuned Weighted Ensemble,COMBINED,0.7,0.739323,0.815837,0.821066,0.853877,0.093894,0.451323,0.548677,0.906106
1,Tuned Weighted Ensemble,DOTA,0.7,0.705817,0.767427,0.774500,0.807464,0.122600,0.485836,0.514164,0.877400


In [ ]:
individual_results_clean = results_df.copy()
individual_results_clean["threshold"] = np.nan

final_comparison_df = pd.concat(
    [individual_results_clean, tuned_ensemble_eval_df],
    ignore_index=True
)

final_comparison_df.sort_values(
    ["validation_set_name", "CV Macro F1"],
    ascending=[True, False]
)

final_comparison_df

,model_name,validation_set_name,CV Macro F1,CV Weighted F1,Accuracy,Precision,FPR,FNR,TPR,TNR,threshold
0,social_media_LinearSVC_(Optuna)_model,WOT,0.758345,0.843587,0.845132,0.868140,0.090771,0.402384,0.597616,0.909229,NaN
1,social_media_LogisticRegressionCV_model,WOT,0.750263,0.837312,0.837840,0.856766,0.099832,0.402849,0.597151,0.900168,NaN
2,social_media_LinearSVC_(Optuna)_model,COMBINED,0.732013,0.808306,0.811067,0.834699,0.111510,0.436923,0.563077,0.888490,NaN
3,social_media_LogisticRegressionCV_model,COMBINED,0.721943,0.800600,0.802931,0.822841,0.119109,0.446784,0.553216,0.880891,NaN
4,social_media_LinearSVC_(Optuna)_model,DOTA,0.698087,0.758817,0.763113,0.785798,0.143867,0.472227,0.527773,0.856133,NaN
5,social_media_LogisticRegressionCV_model,DOTA,0.685568,0.749048,0.753788,0.773102,0.149184,0.491692,0.508308,0.850816,NaN
6,social_media_ComplementNB_model,WOT-TF-IDF-ONLY,0.669180,0.757852,0.737779,0.733877,0.248938,0.313516,0.686484,0.751062,NaN
7,social_media_ComplementNB_model,COMBINED-TF-IDF-ONLY,0.649935,0.718770,0.699201,0.697664,0.295121,0.318986,0.681014,0.704879,NaN
8,social_media_ComplementNB_model,DOTA-TF-IDF-ONLY,0.618675,0.662013,0.644894,0.647828,0.367173,0.324577,0.675423,0.632827,NaN
9,Tuned Weighted Ensemble,WOT,0.765637,0.850411,0.854145,0.885255,0.075495,0.417557,0.582443,0.924505,0.7


In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit

def get_model_confidences_for_validation_set(
    social_media_models,
    validation_set_name,
    validation_sets_scaled,
    validation_sets_nb,
):
    X_scaled, y_val = validation_sets_scaled[validation_set_name]
    X_nb, _ = validation_sets_nb[f"{validation_set_name}-TF-IDF-ONLY"]

    model_names = []
    confidences = []

    for model_name, model in social_media_models.items():

        if "ComplementNB" in model_name:
            X_use = X_nb
        else:
            X_use = X_scaled

        if hasattr(model, "predict_proba"):
            toxic_confidence = model.predict_proba(X_use)[:, 1]

        elif hasattr(model, "decision_function"):
            raw_scores = model.decision_function(X_use)
            toxic_confidence = expit(raw_scores)

        else:
            toxic_confidence = model.predict(X_use)

        model_names.append(model_name)
        confidences.append(toxic_confidence)

    confidences = np.vstack(confidences).T  # shape: n_samples x n_models

    return model_names, confidences, y_val



In [ ]:
def evaluate_confidence_average_ensemble(
    social_media_models,
    validation_sets_scaled,
    validation_sets_nb,
    threshold=0.5,
    validation_set_names=("WOT", "DOTA", "COMBINED"),
):
    res = {
        "ensemble_name": [],
        "validation_set_name": [],
        "threshold": [],
        "CV Macro F1": [],
        "CV Weighted F1": [],
        "Accuracy": [],
        "Precision": [],
        "FPR": [],
        "FNR": [],
        "TPR": [],
        "TNR": [],
    }

    for validation_set_name in validation_set_names:
        model_names, confidences, y_val = get_model_confidences_for_validation_set(
            social_media_models=social_media_models,
            validation_set_name=validation_set_name,
            validation_sets_scaled=validation_sets_scaled,
            validation_sets_nb=validation_sets_nb,
        )

        avg_confidence = confidences.mean(axis=1)
        y_pred = (avg_confidence >= threshold).astype(int)

        metrics = _prediction_metrics(y_val, y_pred)

        res["ensemble_name"].append("Average Confidence Ensemble")
        res["validation_set_name"].append(validation_set_name)
        res["threshold"].append(threshold)

        for metric_name, metric_value in metrics.items():
            res[metric_name].append(metric_value)

    return pd.DataFrame(res)

In [ ]:
confidence_ensemble_df = evaluate_confidence_average_ensemble(
    social_media_models=social_media_models,
    validation_sets_scaled=validation_sets_scaled,
    validation_sets_nb=validation_sets_nb,
    threshold=0.5,
)

confidence_ensemble_df.sort_values(by=['CV Macro F1', 'CV Weighted F1', 'Accuracy'], ascending=False, inplace=True)
confidence_ensemble_df

,ensemble_name,validation_set_name,threshold,CV Macro F1,CV Weighted F1,Accuracy,Precision,FPR,FNR,TPR,TNR
0,Average Confidence Ensemble,WOT,0.5,0.752882,0.838360,0.838222,0.856028,0.102438,0.390927,0.609073,0.897562
2,Average Confidence Ensemble,COMBINED,0.5,0.720879,0.797781,0.797997,0.813101,0.131569,0.427610,0.572390,0.868431
1,Average Confidence Ensemble,DOTA,0.5,0.679867,0.740684,0.741370,0.751348,0.177019,0.465105,0.534895,0.822981


In [ ]:
def evaluate_weighted_confidence_ensemble(
    social_media_models,
    validation_sets_scaled,
    validation_sets_nb,
    weights,
    threshold=0.5,
    validation_set_names=("WOT", "DOTA", "COMBINED"),
):
    res = {
        "ensemble_name": [],
        "validation_set_name": [],
        "threshold": [],
        "CV Macro F1": [],
        "CV Weighted F1": [],
        "Accuracy": [],
        "Precision": [],
        "FPR": [],
        "FNR": [],
        "TPR": [],
        "TNR": [],
    }

    for validation_set_name in validation_set_names:
        model_names, confidences, y_val = get_model_confidences_for_validation_set(
            social_media_models=social_media_models,
            validation_set_name=validation_set_name,
            validation_sets_scaled=validation_sets_scaled,
            validation_sets_nb=validation_sets_nb,
        )

        weight_vector = np.array([weights[name] for name in model_names])
        weight_vector = weight_vector / weight_vector.sum()

        ensemble_confidence = confidences @ weight_vector
        y_pred = (ensemble_confidence >= threshold).astype(int)

        metrics = _prediction_metrics(y_val, y_pred)

        res["ensemble_name"].append("Weighted Confidence Ensemble")
        res["validation_set_name"].append(validation_set_name)
        res["threshold"].append(threshold)

        for metric_name, metric_value in metrics.items():
            res[metric_name].append(metric_value)

    return pd.DataFrame(res)

In [ ]:
weights = {
    "social_media_ComplementNB_model": 0.20,
    "social_media_LinearSVC_(Optuna)_model": 0.45,
    "social_media_LogisticRegressionCV_model": 0.35,
}

weighted_confidence_df = evaluate_weighted_confidence_ensemble(
    social_media_models=social_media_models,
    validation_sets_scaled=validation_sets_scaled,
    validation_sets_nb=validation_sets_nb,
    weights=weights,
    threshold=0.5,
)

weighted_confidence_df.sort_values(by=['CV Macro F1', 'CV Weighted F1', 'Accuracy'], ascending=False, inplace=True)
weighted_confidence_df

,ensemble_name,validation_set_name,threshold,CV Macro F1,CV Weighted F1,Accuracy,Precision,FPR,FNR,TPR,TNR
0,Weighted Confidence Ensemble,WOT,0.5,0.755110,0.840631,0.841311,0.861473,0.096945,0.397120,0.602880,0.903055
2,Weighted Confidence Ensemble,COMBINED,0.5,0.724482,0.801652,0.803154,0.821839,0.122456,0.435123,0.564877,0.877544
1,Weighted Confidence Ensemble,DOTA,0.5,0.685319,0.746886,0.749440,0.764261,0.162257,0.473967,0.526033,0.837743


In [ ]:
def tune_weighted_confidence_ensemble(
    social_media_models,
    validation_sets_scaled,
    validation_sets_nb,
    validation_set_name="COMBINED",
    weight_step=0.05,
    thresholds=None,
    sort_metric="CV Macro F1",
):
    if thresholds is None:
        thresholds = np.arange(0.10, 0.91, 0.01)

    model_names, confidences, y_val = get_model_confidences_for_validation_set(
        social_media_models=social_media_models,
        validation_set_name=validation_set_name,
        validation_sets_scaled=validation_sets_scaled,
        validation_sets_nb=validation_sets_nb,
    )

    if len(model_names) != 3:
        raise ValueError("This function currently assumes exactly 3 models.")

    weight_values = np.arange(0, 1 + weight_step, weight_step)
    results = []

    for w1 in weight_values:
        for w2 in weight_values:
            w3 = 1 - w1 - w2

            if w3 < 0:
                continue

            weights = np.array([w1, w2, w3])

            if not np.isclose(weights.sum(), 1.0):
                continue

            ensemble_confidence = confidences @ weights

            for threshold in thresholds:
                y_pred = (ensemble_confidence >= threshold).astype(int)
                metrics = _prediction_metrics(y_val, y_pred)

                row = {
                    "validation_set_name": validation_set_name,
                    "threshold": round(float(threshold), 3),
                }

                for model_name, weight in zip(model_names, weights):
                    row[f"weight__{model_name}"] = round(float(weight), 3)

                row.update(metrics)
                results.append(row)

    results_df = pd.DataFrame(results)

    return (
        results_df
        .sort_values(sort_metric, ascending=False)
        .reset_index(drop=True)
    )

In [ ]:
tuned_confidence_df = tune_weighted_confidence_ensemble(
    social_media_models=social_media_models,
    validation_sets_scaled=validation_sets_scaled,
    validation_sets_nb=validation_sets_nb,
    validation_set_name="COMBINED",
    weight_step=0.05,
    thresholds=np.arange(0.10, 0.91, 0.01),
    sort_metric="CV Macro F1",
)

tuned_confidence_df.head(10)

,validation_set_name,threshold,weight__social_media_ComplementNB_model,weight__social_media_LinearSVC_(Optuna)_model,weight__social_media_LogisticRegressionCV_model,CV Macro F1,CV Weighted F1,Accuracy,Precision,FPR,FNR,TPR,TNR
0,COMBINED,0.74,0.45,0.55,0.00,0.748185,0.831451,0.848400,0.939453,0.029466,0.542808,0.457192,0.970534
1,COMBINED,0.74,0.50,0.50,0.00,0.748055,0.831335,0.848251,0.939001,0.029710,0.542651,0.457349,0.970290
2,COMBINED,0.75,0.40,0.50,0.10,0.747527,0.831021,0.848027,0.938958,0.029661,0.543747,0.456253,0.970339
3,COMBINED,0.75,0.45,0.50,0.05,0.747374,0.831103,0.848381,0.940915,0.028513,0.545938,0.454062,0.971487
4,COMBINED,0.74,0.50,0.45,0.05,0.747316,0.830638,0.847320,0.936066,0.031323,0.541399,0.458601,0.968677
5,COMBINED,0.75,0.45,0.55,0.00,0.747127,0.831069,0.848549,0.942188,0.027755,0.547660,0.452340,0.972245
6,COMBINED,0.75,0.50,0.45,0.05,0.747076,0.830855,0.848083,0.940078,0.028953,0.545782,0.454218,0.971047
7,COMBINED,0.75,0.55,0.40,0.05,0.747052,0.830822,0.848027,0.939860,0.029075,0.545625,0.454375,0.970925
8,COMBINED,0.74,0.55,0.40,0.05,0.747003,0.830398,0.847059,0.935450,0.031640,0.541478,0.458522,0.968360
9,COMBINED,0.75,0.60,0.35,0.05,0.746840,0.830648,0.847822,0.939300,0.029368,0.545547,0.454453,0.970632


In [ ]:
tuned_confidence_df[tuned_confidence_df["FPR"] <= 0.25].head(10)

,validation_set_name,threshold,weight__social_media_ComplementNB_model,weight__social_media_LinearSVC_(Optuna)_model,weight__social_media_LogisticRegressionCV_model,CV Macro F1,CV Weighted F1,Accuracy,Precision,FPR,FNR,TPR,TNR
0,COMBINED,0.74,0.45,0.55,0.00,0.748185,0.831451,0.848400,0.939453,0.029466,0.542808,0.457192,0.970534
1,COMBINED,0.74,0.50,0.50,0.00,0.748055,0.831335,0.848251,0.939001,0.029710,0.542651,0.457349,0.970290
2,COMBINED,0.75,0.40,0.50,0.10,0.747527,0.831021,0.848027,0.938958,0.029661,0.543747,0.456253,0.970339
3,COMBINED,0.75,0.45,0.50,0.05,0.747374,0.831103,0.848381,0.940915,0.028513,0.545938,0.454062,0.971487
4,COMBINED,0.74,0.50,0.45,0.05,0.747316,0.830638,0.847320,0.936066,0.031323,0.541399,0.458601,0.968677
5,COMBINED,0.75,0.45,0.55,0.00,0.747127,0.831069,0.848549,0.942188,0.027755,0.547660,0.452340,0.972245
6,COMBINED,0.75,0.50,0.45,0.05,0.747076,0.830855,0.848083,0.940078,0.028953,0.545782,0.454218,0.971047
7,COMBINED,0.75,0.55,0.40,0.05,0.747052,0.830822,0.848027,0.939860,0.029075,0.545625,0.454375,0.970925
8,COMBINED,0.74,0.55,0.40,0.05,0.747003,0.830398,0.847059,0.935450,0.031640,0.541478,0.458522,0.968360
9,COMBINED,0.75,0.60,0.35,0.05,0.746840,0.830648,0.847822,0.939300,0.029368,0.545547,0.454453,0.970632


In [ ]:
# assemble the best model with weights and threshold from the tuning results

class TunedWeightedConfidenceEnsemble:
    def __init__(self, social_media_models, weights, threshold):
        self.social_media_models = social_media_models
        self.weights = weights
        self.threshold = threshold

    def predict(self, X_scaled, X_nb):
        model_names = []
        confidences = []

        for model_name, model in self.social_media_models.items():

            if "ComplementNB" in model_name:
                X_use = X_nb
            else:
                X_use = X_scaled

            if hasattr(model, "predict_proba"):
                toxic_confidence = model.predict_proba(X_use)[:, 1]

            elif hasattr(model, "decision_function"):
                raw_scores = model.decision_function(X_use)
                toxic_confidence = expit(raw_scores)

            else:
                toxic_confidence = model.predict(X_use)

            model_names.append(model_name)
            confidences.append(toxic_confidence)

        confidences = np.vstack(confidences).T  # shape: n_samples x n_models

        weight_vector = np.array([self.weights[name] for name in model_names])
        weight_vector = weight_vector / weight_vector.sum()

        ensemble_confidence = confidences @ weight_vector
        y_pred = (ensemble_confidence >= self.threshold).astype(int)

        return y_pred
    
    
best_weights_confidence = {
    "social_media_ComplementNB_model": tuned_confidence_df[tuned_confidence_df["FPR"] <= 0.25].iloc[0]["weight__social_media_ComplementNB_model"],
    "social_media_LinearSVC_(Optuna)_model": tuned_confidence_df[tuned_confidence_df["FPR"] <= 0.25].iloc[0]["weight__social_media_LinearSVC_(Optuna)_model"],
    "social_media_LogisticRegressionCV_model": tuned_confidence_df[tuned_confidence_df["FPR"] <= 0.25].iloc[0]["weight__social_media_LogisticRegressionCV_model"],
}

best_threshold_confidence = tuned_confidence_df[tuned_confidence_df["FPR"] <= 0.25].iloc[0]["threshold"]

best_confidence_ensemble = TunedWeightedConfidenceEnsemble(
    social_media_models=social_media_models,
    weights=best_weights_confidence,
    threshold=best_threshold_confidence,
)

# Test on the hold-off data

In [ ]:
# Evaluate the best model on the test set

wot_val_nb = nb_tfidf.transform(wot_val_text)
wot_val_nb_scaled = scaler.transform(wot_val_nb)

dota_val_nb = nb_tfidf.transform(dota_val_text)
dota_val_nb_scaled = scaler.transform(dota_val_nb)


combined_val_nb = vstack([wot_val_nb, dota_val_nb])
combined_val_nb_scaled = vstack([wot_val_nb_scaled, dota_val_nb_scaled])

y_test_pred = best_confidence_ensemble.predict(combined_val_nb, combined_val_nb_scaled)

NameError: name 'nb_tfidf' is not defined

In [ ]:
# evaluate predictions
y_test_true = np.hstack([wot_val_y_binary, dota_val_y_binary])

test_metrics = _prediction_metrics(y_test_true, y_test_pred)
test_metrics

{'CV Macro F1': 0.7133600705314467,
 'CV Weighted F1': 0.8233082015105148,
 'Accuracy': 0.8452157598499062,
 'Precision': np.float64(0.9346131011533376),
 'FPR': np.float64(0.026889371582652984),
 'FNR': np.float64(0.6156545209176788),
 'TPR': np.float64(0.3843454790823212),
 'TNR': np.float64(0.973110628417347)}

# Test Model Object 

In [ ]:
from src.model.social_media_collection import SocialMediaModelCollection
from src.ensemble.ensemble import Ensemble

paths = list((CONFIG['model_dir'] / 'binary' ).glob('social_media_*.joblib'))

scaler_path = CONFIG['model_dir'] / 'helper' / 'social_media_max_abs_scaler.joblib'
nb_tfidf_path = CONFIG['model_dir'] / 'helper' / 'social_media_nb_tfidf.joblib'

In [ ]:
social_media_collection = SocialMediaModelCollection(
    model_joblibs=paths,
    scaler_path=scaler_path,
    nb_tfidf_path=nb_tfidf_path,
)

In [ ]:
social_media_collection.classifiers

{PosixPath('/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_ComplementNB_model.joblib'): Pipeline(steps=[('clf', ComplementNB(alpha=1.2006196407511314))]),
 PosixPath('/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_LinearSVC_(Optuna)_model.joblib'): Pipeline(steps=[('clf',
                  LinearSVC(C=0.7813882258601701, class_weight='balanced',
                            dual=False, max_iter=5000, penalty='l1',
                            random_state=42))]),
 PosixPath('/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_LogisticRegressionCV_model.joblib'): Pipeline(steps=[('clf',
                  LogisticRegressionCV(class_weight='balanced',
                                       cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                                       max_iter=1000, n_

In [ ]:
# create ensemble object 
ensemble = Ensemble(model_collections=[social_media_collection])
ensemble

In [ ]:
thresholds = np.arange(0.50, 0.95, 0.01)

In [ ]:
def _safe_confusion_rates(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'FPR': fp / (fp + tn) if (fp + tn) > 0 else 0,
        'FNR': fn / (fn + tp) if (fn + tp) > 0 else 0,
        'TPR': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'TNR': tn / (tn + fp) if (tn + fp) > 0 else 0,
    }
    
def score_func(y_true, y_pred):
    fpr = _safe_confusion_rates(y_true, y_pred)['FPR']
    return f1_score(y_true, y_pred, average='macro') 

def precision(y_true, y_pred):
    rates = _safe_confusion_rates(y_true, y_pred)
    return rates['TPR'] / (rates['TPR'] + rates['FPR']) if (rates['TPR'] + rates['FPR']) > 0 else 0

def _prediction_metrics(y_true, y_pred):
    return {
        'CV Macro F1': f1_score(y_true, y_pred, average='macro'),
        'CV Weighted F1': f1_score(y_true, y_pred, average='weighted'),
        'Accuracy': accuracy_score(y_true, y_pred),
        "Precision": precision(y_true, y_pred),
        **_safe_confusion_rates(y_true, y_pred),
    }


In [ ]:
dota_pred = ensemble.fit_weighted_confidence_majority(X_val=dota_train_x[tc], y_val=dota_train_y_binary, score_func=score_func, thresholds=thresholds)

In [ ]:
wot_pred = ensemble.fit_weighted_confidence_majority(X_val=wot_train_x[tc], y_val=wot_train_y_binary, score_func=score_func, thresholds=thresholds)

In [ ]:
combined_val_x = pd.concat([wot_train_x[tc], dota_train_x[tc]], ignore_index=True)
combined_val_y = np.hstack([wot_train_y_binary, dota_train_y_binary])

combined_val_pred = ensemble.fit_weighted_confidence_majority(X_val=combined_val_x, y_val=combined_val_y, score_func=score_func, thresholds=thresholds)

In [ ]:
combined_val_pred

(array([0.06810403, 0.91104057, 0.0208554 ]), 0.732572284771511)

In [ ]:
combined_val_x = pd.concat([wot_train_x[tc], dota_train_x[tc]], ignore_index=True)
combined_val_y = np.hstack([wot_train_y_binary, dota_train_y_binary])

In [ ]:
combined_pred = ensemble.predict_weighted_confidence_majority(X=combined_val_x)

In [ ]:
_prediction_metrics(combined_val_y, combined_pred)

{'CV Macro F1': 0.732572284771511,
 'CV Weighted F1': 0.8086142750387438,
 'Accuracy': 0.8112722736328598,
 'Precision': np.float64(0.8347767271130508),
 'FPR': np.float64(0.11180336680593222),
 'FNR': np.float64(0.43512286742839257),
 'TPR': np.float64(0.5648771325716074),
 'TNR': np.float64(0.8881966331940678)}